# ROCm-Optimized Multi-Label Milk Adulteration & SHAP Interpretability Pipeline

## Overview
This notebook implements an end-to-end computer vision pipeline for identifying singular and compound milk adulterations using handcrafted vision features and explainable AI techniques.

## Feature Sets
- **RGB Color Space Metrics**: Channel means and variances capturing intensity distributions
- **GLCM Texture Features**: Gray-Level Co-occurrence Matrix properties (contrast, dissimilarity, homogeneity, energy, correlation) encoding spatial texture patterns
- **LBP Microstructure**: Local Binary Patterns capturing local texture variations and distribution statistics

## Target Classification
Multi-label classification for milk adulterations:
- **Authentic**: Unadulterated milk
- **Detergent Added**: Presence of detergent additives
- **Urea Added**: Urea contamination
- **Water Diluted**: Water dilution

Compound targets (combinations) are detected from YOLO-style label annotations.

## Interpretability
SHAP (SHapley Additive exPlanations) provides global and local feature importance analysis, replacing deep learning visualizers with model-agnostic explanations compatible with ROCm-optimized environments.

In [ ]:
# ============================================================================
# CELL 2: Imports & Environment Setup
# ============================================================================

import os
import cv2
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, hamming_loss, accuracy_score

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.color import rgb2gray

import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Path Declarations (Linux environment - adjust for cross-platform compatibility)
# ============================================================================

# Original Windows paths mapped to Linux workspace structure
BASE_PATH = Path('//food-guard-ai')
IMAGE_DIR = BASE_PATH / 'data' / 'milk_evaporative_dataset' / 'images' / 'train'
LABEL_DIR = BASE_PATH / 'data' / 'milk_evaporative_dataset' / 'labels' / 'train'

# Verify directories exist
assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"
assert LABEL_DIR.exists(), f"Label directory not found: {LABEL_DIR}"

print(f"Image Directory: {IMAGE_DIR}")
print(f"Label Directory: {LABEL_DIR}")
print(f"Images found: {len(list(IMAGE_DIR.glob('*.*')))}")
print(f"Labels found: {len(list(LABEL_DIR.glob('*.txt')))}")

# Class definitions
CLASS_NAMES = ['authentic', 'detergent_added', 'urea_added', 'water_diluted']
NUM_CLASSES = len(CLASS_NAMES)

print(f"\nTarget Classes: {CLASS_NAMES}")

In [ ]:
# ============================================================================
# CELL 3: Multi-Label Dataset Parser
# ============================================================================

def parse_yolo_label_file(label_path):
    """
    Parse YOLO-style label file and extract class assignments.
    Returns a binary vector for multi-label classification.

    Args:
        label_path: Path to .txt label file

    Returns:
        Binary numpy array of shape (NUM_CLASSES,) where 1 indicates class presence
    """
    label_vector = np.zeros(NUM_CLASSES, dtype=np.int8)

    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                class_id = int(line.strip().split()[0])
                if 0 <= class_id < NUM_CLASSES:
                    label_vector[class_id] = 1
    except Exception as e:
        print(f"Error parsing {label_path}: {e}")

    return label_vector


def build_multilabel_dataset():
    """
    Scan image and label directories to build a multi-label dataset.
    Returns a DataFrame with image paths and binary label columns.
    """
    data_records = []

    # Get all image files
    image_files = sorted([
        f for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ])

    print(f"Processing {len(image_files)} images...")

    for img_file in image_files:
        img_path = IMAGE_DIR / img_file

        # Construct label file path (remove extension, add .txt)
        base_name = os.path.splitext(img_file)[0]
        label_file = base_name + '.txt'
        label_path = LABEL_DIR / label_file

        if label_path.exists():
            label_vector = parse_yolo_label_file(label_path)

            record = {
                'image_path': str(img_path),
                'label_file': str(label_path)
            }

            # Add individual class columns
            for class_idx, class_name in enumerate(CLASS_NAMES):
                record[class_name] = label_vector[class_idx]

            data_records.append(record)

    df = pd.DataFrame(data_records)

    print(f"\nDataset Summary:")
    print(f"Total images with labels: {len(df)}")
    print(f"\nClass Distribution:")
    for class_name in CLASS_NAMES:
        count = df[class_name].sum()
        pct = (count / len(df)) * 100
        print(f"  {class_name}: {int(count)} ({pct:.1f}%)")

    print(f"\nMulti-label Statistics:")
    df['num_labels'] = df[CLASS_NAMES].sum(axis=1)
    print(df['num_labels'].value_counts().sort_index())

    return df


# Build dataset
dataset_df = build_multilabel_dataset()
print(f"\nFirst few records:")
print(dataset_df.head())

In [ ]:
# ============================================================================
# CELL 4: Handcrafted Vision Feature Extraction Engine
# ============================================================================

def extract_rgb_features(image):
    """
    Extract RGB color space metrics: channel means and variances.
    Returns 6 features: [R_mean, G_mean, B_mean, R_var, G_var, B_var]
    """
    features = []

    # Ensure 3-channel RGB
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)

    # Normalize to [0, 1] for stability
    image = image.astype(np.float32) / 255.0

    # Extract RGB channel means and variances
    for channel_idx in range(3):
        channel_data = image[:, :, channel_idx]
        features.append(np.mean(channel_data))

    for channel_idx in range(3):
        channel_data = image[:, :, channel_idx]
        features.append(np.var(channel_data))

    return np.array(features)


def extract_glcm_features(image, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    """
    Extract Gray-Level Co-occurrence Matrix (GLCM) texture features.
    Returns GLCM properties: contrast, dissimilarity, homogeneity, energy, correlation
    """
    # Convert to grayscale
    if len(image.shape) == 3:
        gray = rgb2gray(image)
    else:
        gray = image.astype(np.float32) / 255.0

    # Quantize to 64 levels for computational efficiency
    gray_quantized = (gray * 63).astype(np.uint8)

    # Compute GLCM
    glcm = graycomatrix(gray_quantized, distances=distances, angles=angles,
                        levels=64, symmetric=True, normed=True)

    # Extract GLCM properties
    properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
    features = []

    for prop in properties:
        prop_values = graycoprops(glcm, prop)
        # Average across all angles and distances
        features.append(np.mean(prop_values))

    return np.array(features)


def extract_lbp_features(image, radius=1, n_points=8):
    """
    Extract Local Binary Pattern (LBP) texture features.
    Returns LBP mean and variance statistics.
    """
    # Convert to grayscale
    if len(image.shape) == 3:
        gray = rgb2gray(image)
    else:
        gray = image.astype(np.float32) / 255.0

    # Quantize to 8-bit
    gray_uint8 = (gray * 255).astype(np.uint8)

    # Compute LBP
    lbp = local_binary_pattern(gray_uint8, n_points, radius, method='uniform')

    # Extract statistics
    lbp_mean = np.mean(lbp)
    lbp_var = np.var(lbp)

    return np.array([lbp_mean, lbp_var])


def extract_all_features(image_path):
    """
    Extract all handcrafted vision features from an image.
    Total: 6 (RGB) + 5 (GLCM) + 2 (LBP) = 13 features
    """
    try:
        # Load image
        image = cv2.imread(str(image_path))
        if image is None:
            print(f"Warning: Could not load image {image_path}")
            return None

        # Extract feature components
        rgb_features = extract_rgb_features(image)
        glcm_features = extract_glcm_features(image)
        lbp_features = extract_lbp_features(image)

        # Concatenate all features
        all_features = np.concatenate([rgb_features, glcm_features, lbp_features])

        return all_features

    except Exception as e:
        print(f"Error extracting features from {image_path}: {e}")
        return None


# Feature extraction for entire dataset
print("Extracting vision features from all images...")
feature_list = []
valid_indices = []

for idx, row in dataset_df.iterrows():
    features = extract_all_features(row['image_path'])
    if features is not None:
        feature_list.append(features)
        valid_indices.append(idx)

    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(dataset_df)} images")

# Create feature matrix X and label matrix Y
X = np.array(feature_list)
Y = dataset_df.loc[valid_indices, CLASS_NAMES].values

print(f"\nFeature Extraction Complete:")
print(f"Feature matrix shape: {X.shape}")
print(f"Label matrix shape: {Y.shape}")
print(f"Features per image: {X.shape[1]}")
print(f"  - RGB features: 6")
print(f"  - GLCM features: 5")
print(f"  - LBP features: 2")

# Feature statistics
print(f"\nFeature Statistics:")
print(f"  Mean: {X.mean(axis=0).round(4)}")
print(f"  Std: {X.std(axis=0).round(4)}")

In [ ]:
# ============================================================================
# CELL 5: Model Training & Multi-Label Evaluation
# ============================================================================

# Feature standardization for better model performance
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
test_size = 0.2
random_state = 42

X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y, test_size=test_size, random_state=random_state, stratify=Y.sum(axis=1)
)

print(f"Dataset Split:")
print(f"  Training set: {X_train.shape[0]} samples")
print(f"  Testing set: {X_test.shape[0]} samples")
print(f"  Test ratio: {test_size*100:.1f}%")

# Multi-label classification model
base_estimator = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=random_state,
    n_jobs=-1,
    verbose=0
)

model = MultiOutputClassifier(base_estimator)

print(f"\nTraining Multi-Output Random Forest Classifier...")
model.fit(X_train, Y_train)
print(f"Training complete!")

# Predictions
Y_pred_train = model.predict(X_train)
Y_pred_test = model.predict(X_test)

# Multi-label evaluation metrics
hamming_loss_train = hamming_loss(Y_train, Y_pred_train)
hamming_loss_test = hamming_loss(Y_test, Y_pred_test)

# Exact match accuracy
exact_match_train = accuracy_score(Y_train, Y_pred_train)
exact_match_test = accuracy_score(Y_test, Y_pred_test)

print(f"\n" + "="*70)
print(f"MULTI-LABEL CLASSIFICATION RESULTS")
print(f"="*70)

print(f"\nHamming Loss (fraction of incorrect labels):")
print(f"  Training: {hamming_loss_train:.4f}")
print(f"  Testing:  {hamming_loss_test:.4f}")

print(f"\nExact Match Accuracy (all labels correct):")
print(f"  Training: {exact_match_train:.4f}")
print(f"  Testing:  {exact_match_test:.4f}")

print(f"\n" + "-"*70)
print(f"Per-Class Performance (Testing Set):")
print(f"-"*70)

for class_idx, class_name in enumerate(CLASS_NAMES):
    # Binary classification report for each class
    print(f"\n{class_name.upper()}:")
    report = classification_report(
        Y_test[:, class_idx],
        Y_pred_test[:, class_idx],
        target_names=['Absent', 'Present'],
        digits=4
    )
    print(report)

In [ ]:
# ============================================================================
# CELL 6: Global & Local SHAP Explanations
# ============================================================================

print(f"Initializing SHAP TreeExplainer for model interpretation...\n")

# Create SHAP explainer for each class estimator
shap_explainers = []
shap_values_list = []

for class_idx, class_name in enumerate(CLASS_NAMES):
    print(f"Computing SHAP values for {class_name}...")

    # Get the individual estimator for this class
    estimator = model.estimators_[class_idx]

    # Create TreeExplainer
    explainer = shap.TreeExplainer(estimator)
    shap_explainers.append(explainer)

    # Compute SHAP values for test set
    shap_vals = explainer.shap_values(X_test)
    shap_values_list.append(shap_vals)

    print(f"  SHAP shape: {shap_vals.shape}")

print(f"\nSHAP computation complete!\n")

# Define feature names
rgb_names = ['R_mean', 'G_mean', 'B_mean', 'R_var', 'G_var', 'B_var']
glcm_names = ['GLCM_contrast', 'GLCM_dissimilarity', 'GLCM_homogeneity', 'GLCM_energy', 'GLCM_correlation']
lbp_names = ['LBP_mean', 'LBP_var']
feature_names = rgb_names + glcm_names + lbp_names

print(f"Feature Names: {feature_names}\n")

# ============================================================================
# Global SHAP Summary Plots
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('SHAP Global Feature Importance - Multi-Label Adulteration Detection', fontsize=16, fontweight='bold')

for class_idx, class_name in enumerate(CLASS_NAMES):
    ax = axes[class_idx // 2, class_idx % 2]

    # Summary plot (bar plot of mean absolute SHAP values)
    shap_abs_mean = np.abs(shap_values_list[class_idx]).mean(axis=0)
    sorted_idx = np.argsort(shap_abs_mean)[::-1]

    ax.barh(np.array(feature_names)[sorted_idx], shap_abs_mean[sorted_idx])
    ax.set_xlabel('Mean |SHAP Value|', fontweight='bold')
    ax.set_title(f'{class_name.replace("_", " ").title()}', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('//food-guard-ai/ml/shap_global_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Global SHAP summary plots saved!\n")

# ============================================================================
# Feature Importance Analysis
# ============================================================================

print(f"="*70)
print(f"GLOBAL FEATURE IMPORTANCE RANKINGS (SHAP)")
print(f"="*70)

for class_idx, class_name in enumerate(CLASS_NAMES):
    print(f"\n{class_name.upper()}:")
    print("-" * 50)

    shap_abs_mean = np.abs(shap_values_list[class_idx]).mean(axis=0)
    sorted_idx = np.argsort(shap_abs_mean)[::-1]

    for rank, feat_idx in enumerate(sorted_idx[:5], 1):
        feat_name = feature_names[feat_idx]
        importance = shap_abs_mean[feat_idx]
        print(f"  {rank}. {feat_name:30s} = {importance:.6f}")

# ============================================================================
# Local SHAP Explanation for Sample Predictions
# ============================================================================

print(f"\n" + "="*70)
print(f"LOCAL SHAP EXPLANATIONS - SAMPLE PREDICTIONS")
print(f"="*70)

# Select diverse samples: correct and incorrect predictions
sample_indices = [0, 5, 10]  # First few test samples

for sample_idx in sample_indices:
    print(f"\n[Sample {sample_idx}]")
    print("-" * 50)

    print(f"True labels: {', '.join([CLASS_NAMES[i] for i in range(NUM_CLASSES) if Y_test[sample_idx, i]])}")
    print(f"Predicted:   {', '.join([CLASS_NAMES[i] for i in range(NUM_CLASSES) if Y_pred_test[sample_idx, i]])}")

    print(f"\nTop influential features for each class:")
    for class_idx, class_name in enumerate(CLASS_NAMES):
        shap_vals_sample = shap_values_list[class_idx][sample_idx]
        top_idx = np.argsort(np.abs(shap_vals_sample))[::-1][:3]

        print(f"  {class_name}:")
        for rank, feat_idx in enumerate(top_idx, 1):
            feat_val = feature_names[feat_idx]
            shap_val = shap_vals_sample[feat_idx]
            print(f"    {rank}. {feat_val:30s} (SHAP: {shap_val:+.4f})")

print(f"\n" + "="*70)
print(f"Pipeline execution complete. Results and visualizations saved.")
print(f"="*70)